# 2e. Cross-Scenario Validation Summary

Aggregates results from notebooks 2a–2d to answer:
1. How does model ID accuracy degrade across scenarios?
2. Do GS and SBI agree, and does agreement degrade?
3. Which scenarios are problematic?

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

_NOTEBOOK_DIR = Path(os.path.abspath(''))
_NOTEBOOK_ROOT = _NOTEBOOK_DIR.parent
_PROJECT_ROOT = _NOTEBOOK_ROOT.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
if str(_NOTEBOOK_ROOT) not in sys.path:
	sys.path.insert(0, str(_NOTEBOOK_ROOT))

from shared_setup import *
import pickle


## 1. Load Results From 2a–2d

In [ ]:
RESULTS_DIR = Path('../results/validation')

scenarios = {
    '2a: Static Uniform': '2a_static_uniform.pkl',
    '2b: Dynamic Uniform': '2b_dynamic_uniform.pkl',
    '2c: Static Hard-A': '2c_static_hard_a.pkl',
    '2d: Dynamic Hard-A': '2d_dynamic_hard_a.pkl',
}

all_results = {}
for label, fname in scenarios.items():
    path = RESULTS_DIR / fname
    if path.exists():
        with open(path, 'rb') as f:
            all_results[label] = pickle.load(f)
        print(f'Loaded {label}')
    else:
        print(f'Missing: {path} — run notebook {label[:2]} first')

## 2. Per-Scenario Accuracy

In [ ]:
summary_rows = []

for label, data in all_results.items():
    gs = data.get('gs')
    sbi = data.get('sbi')

    row = {'scenario': label}
    if gs is not None and len(gs) > 0:
        row['gs_accuracy'] = gs['gs_correct'].mean()
        row['gs_n'] = len(gs)
    if sbi is not None and len(sbi) > 0:
        row['sbi_accuracy'] = sbi['sbi_correct'].mean()
        row['sbi_n'] = len(sbi)
    if gs is not None and sbi is not None and len(gs) > 0 and len(sbi) > 0:
        merged = gs.merge(sbi, on=['animal_id', 'true_model'])
        row['agreement'] = (merged['gs_winner'] == merged['sbi_winner']).mean()
        row['both_correct'] = (merged['gs_correct'] & merged['sbi_correct']).mean()
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False, float_format='%.2f'))

## 3. Visualisation

In [ ]:
if len(summary) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Accuracy per scenario
    ax = axes[0]
    x = np.arange(len(summary))
    w = 0.35
    if 'gs_accuracy' in summary.columns:
        ax.bar(x - w/2, summary['gs_accuracy'], w, label='GS', color='steelblue', alpha=0.7)
    if 'sbi_accuracy' in summary.columns:
        ax.bar(x + w/2, summary['sbi_accuracy'], w, label='SBI', color='darkorange', alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(summary['scenario'], rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color='grey', ls='--', lw=0.8, label='Chance')
    ax.legend(fontsize=8)
    ax.set_title('Model ID Accuracy by Scenario')

    # Agreement
    ax = axes[1]
    if 'agreement' in summary.columns:
        ax.bar(x, summary['agreement'], color='grey', alpha=0.7, label='GS-SBI agree')
    if 'both_correct' in summary.columns:
        ax.bar(x, summary['both_correct'], color='green', alpha=0.5, label='Both correct')
    ax.set_xticks(x)
    ax.set_xticklabels(summary['scenario'], rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Rate')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.set_title('Method Agreement')

    plt.tight_layout()
    plt.show()

## 4. Conclusion

**If accuracy degrades from 2a → 2d**: The method is sensitive to
the scenario complexity. Check which transitions cause the drop.

**If GS-SBI agreement drops**: The methods disagree under harder
conditions. Use GS as primary (validated in 2f/2g).

**If 2d accuracy is low**: Model ID during active adaptation is
unreliable. 4b's per-phase assignments should be interpreted
with caution.